[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E8%A6%96%E8%A6%BA%E5%8C%96%E8%A1%A8%E6%A0%BC/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_%E6%9C%83%E8%AE%8A%E8%89%B2%E7%9A%84%E7%A9%BA%E6%B0%A3%E5%93%81%E8%B3%AA%E9%80%B1%E5%A0%B1_Styler.ipynb)

# 🎨 做一張會變色的空氣品質週報

> 🎯 **情境**：你看過環境部的空氣品質地圖嗎？綠色代表良好、黃色普通、橘色開始對敏感族群不健康……同樣的數字，**上了顏色**，一眼就知道哪裡要小心。學校要你做一份「本週空氣品質報表」貼在公布欄，今天就用 pandas 的 **Styler**，把一堆數字變成一張**跟官方一樣會變色**的表格，最後匯出成 Excel。

| 素材檔案 | 內容 |
| --- | --- |
| `日空氣品質指標_近一年.csv` | 環境部近一年、全台 80 多個測站的每日 AQI 與 PM2.5 等副指標 |

| 📡 資料來源（政府資料開放平台） | 提供機關 | 下載日期 |
| --- | --- | --- |
| [日空氣品質指標(AQI)](https://data.gov.tw/dataset/40507) | 環境部 | 2026-10-24 |

> ✅ 這些都是**真實的政府開放資料**，依「政府資料開放授權條款－第1版」使用。課堂上統一使用上表日期的快照，所以每個人跑出來的數字都一樣。

## 🗺️ 今日任務地圖

| 關卡 | 要回答的問題 |
| --- | --- |
| ⬜ 任務 1 | 先做出「素顏」的週報 |
| ⬜ 任務 2 | 套上官方的 AQI 顏色 |
| ⬜ 任務 3 | 年度成績單——格式化、資料條、標出最好與最差 |
| ⬜ 任務 4 | 熱力圖——一年 12 個月，哪裡、什麼時候最糟？ |
| ⬜ 任務 5 | 匯出成 Excel，貼到公布欄 |

**怎麼用這本 Notebook**：從上到下按 `Shift + Enter` 一格一格執行。每個任務都會先講「你想知道什麼」，再給程式，最後用 💡 **看懂結果** 告訴你這個數字代表什麼。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    '日空氣品質指標_近一年.csv',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/%E8%A6%96%E8%A6%BA%E5%8C%96%E8%A1%A8%E6%A0%BC/%E8%A6%96%E8%A6%BA%E5%8C%96%E8%A1%A8%E6%A0%BC_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['日空氣品質指標_近一年.csv']


## 任務 1：先做出「素顏」的週報

**白話說明**：挑 12 個測站、最近 7 天，用 `pivot` 排成「測站 × 日期」的表。數字都對，但很難一眼看出重點。

In [2]:
import pandas as pd

aqi = pd.read_csv('日空氣品質指標_近一年.csv', parse_dates=['monitordate'])
sites = ['基隆', '中山', '板橋', '桃園', '新竹', '忠明', '斗六', '臺南', '小港', '屏東', '花蓮', '金門']
last7 = aqi[aqi['monitordate'] > aqi['monitordate'].max() - pd.Timedelta(days=7)]     # 最近 7 天

week = last7[last7['sitename'].isin(sites)].pivot(index='sitename', columns='monitordate', values='aqi')
week = week.reindex(sites)                                            # 依我們指定的順序（北 → 南 → 東 → 離島）
week.columns = week.columns.strftime('%m/%d')
week.index.name = '測站'
week

monitordate,10/17,10/18,10/19,10/20,10/21,10/22,10/23
測站,,,,,,,
基隆,28,29,24,22,46,65,65
中山,36,44,33,48,38,50,34
板橋,29,84,45,50,38,48,33
桃園,52,126,48,58,52,45,40
新竹,112,86,57,93,52,40,39
忠明,90,80,100,101,63,80,37
斗六,87,84,84,100,108,94,78
臺南,62,66,69,61,86,115,62
小港,84,70,70,63,82,110,60


## 任務 2：套上官方的 AQI 顏色

**白話說明**：`style.map(函式)` 會把**每一格**的值丟進函式，函式回傳一段 CSS（例如 `'background-color: #00e400'`），那一格就會變色。顏色用的是環境部公告的 AQI 等級色。

In [3]:
def aqi_color(v):
    """依環境部 AQI 等級回傳背景色（文字顏色也一起調整，深色底用白字）。"""
    if pd.isna(v):
        return 'background-color: #eeeeee; color: #999999'
    levels = [(50, '#00e400', 'black'), (100, '#ffff00', 'black'), (150, '#ff7e00', 'black'),
              (200, '#ff0000', 'white'), (300, '#8f3f97', 'white')]
    for upper, bg, fg in levels:
        if v <= upper:
            return f'background-color: {bg}; color: {fg}'
    return 'background-color: #7e0023; color: white'

styled = (week.style
          .map(aqi_color)                                             # 每一格依 AQI 上色
          .format('{:.0f}', na_rep='—')                               # 整數顯示；沒資料顯示「—」
          .set_caption('本週空氣品質 AQI（綠＝良好、黃＝普通、橘＝敏感族群不健康、紅＝不健康）'))
styled

monitordate,10/17,10/18,10/19,10/20,10/21,10/22,10/23
測站,,,,,,,
基隆,28,29,24,22,46,65,65
中山,36,44,33,48,38,50,34
板橋,29,84,45,50,38,48,33
桃園,52,126,48,58,52,45,40
新竹,112,86,57,93,52,40,39
忠明,90,80,100,101,63,80,37
斗六,87,84,84,100,108,94,78
臺南,62,66,69,61,86,115,62
小港,84,70,70,63,82,110,60


> 💡 **看懂結果**：一上色，馬上看出**中南部（斗六、小港、屏東、臺南）黃色、橘色最多**，花蓮、中山大多是綠色；北部偶爾也會冒出一天橘色——那通常是一陣外來的污染。一眼就看出哪裡、哪天要小心。這就是為什麼官方要規定一套顏色——**顏色本身就是資訊**。

## 任務 3：年度成績單——格式化、資料條、標出最好與最差

**白話說明**：
- `format` 設定顯示格式：`'{:.1f}'` 小數一位、`'{:.0%}'` 百分比。
- `bar` 在格子裡畫**資料條**，長短代表大小。
- `highlight_max` / `highlight_min` 標出每一欄的最大、最小值。

In [4]:
year = aqi[aqi['sitename'].isin(sites)].groupby('sitename').agg(
    平均AQI=('aqi', 'mean'),
    最差AQI=('aqi', 'max'),
    良好天數比例=('aqi', lambda s: (s <= 50).mean()),
    不健康天數=('aqi', lambda s: (s > 100).sum()),
).reindex(sites)
year.index.name = '測站'

report = (year.style
          .format({'平均AQI': '{:.1f}', '最差AQI': '{:.0f}', '良好天數比例': '{:.0%}', '不健康天數': '{:.0f} 天'})
          .bar(subset=['不健康天數'], color='#ff7e00', vmin=0)                     # 資料條
          .highlight_max(subset=['平均AQI'], color='#ffcccc')                     # 平均最差：淡紅
          .highlight_min(subset=['平均AQI'], color='#ccffcc')                     # 平均最好：淡綠
          .set_caption('近一年各測站空氣品質成績單'))
report

,平均AQI,最差AQI,良好天數比例,不健康天數
測站,,,,
基隆,50.8,140,56%,6 天
中山,57.4,134,26%,7 天
板橋,57.6,161,36%,14 天
桃園,55.4,140,46%,13 天
新竹,53.8,151,50%,9 天
忠明,59.8,143,35%,17 天
斗六,75.0,154,23%,69 天
臺南,65.9,159,25%,28 天
小港,73.7,132,8%,38 天


> 💡 **看懂結果**：資料條讓「不健康天數」的差距一目了然：斗六、金門的橘色長條，是花蓮的好幾十倍。**花蓮一整年有八成以上的天數是「良好」**，斗六、金門只有兩成多。

## 任務 4：熱力圖——一年 12 個月，哪裡、什麼時候最糟？

**白話說明**：`background_gradient` 依數值大小自動上漸層色，就是「熱力圖」。`axis=None` 代表**整張表一起比較**。

In [5]:
aqi['月份'] = aqi['monitordate'].dt.strftime('%Y-%m')
heat = aqi[aqi['sitename'].isin(sites)].pivot_table(index='sitename', columns='月份', values='aqi', aggfunc='mean').reindex(sites)
heat.index.name = '測站'

heat_style = (heat.style
              .background_gradient(cmap='YlOrRd', axis=None)                     # 整張表一起比：越紅越差
              .format('{:.0f}')
              .set_caption('各測站每月平均 AQI 熱力圖'))
heat_style

月份,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09,2026-10
測站,,,,,,,,,,,,,,
基隆,40,40,55,53,56,59,57,61,53,49,38,39,48,53
中山,57,49,61,58,65,66,63,66,57,61,45,48,51,52
板橋,87,49,58,56,60,66,61,71,58,66,48,46,50,55
桃園,55,46,52,52,58,63,59,63,55,59,48,55,52,63
新竹,50,46,52,51,58,63,59,67,52,47,45,45,52,65
忠明,55,65,56,57,61,67,65,68,63,45,39,47,68,80
斗六,62,76,81,83,86,88,86,79,74,47,43,50,89,99
臺南,44,64,77,80,82,87,73,69,55,40,40,45,64,80
小港,52,70,86,90,90,88,81,74,64,55,54,61,61,82


> 💡 **看懂結果**：熱力圖同時告訴你兩件事——**橫著看**：每個測站都是冬春紅、夏天黃；**直著看**：中南部和金門的紅色最深。一張表抵十張折線圖。

## 任務 5：匯出成 Excel，貼到公布欄

**白話說明**：Styler 可以直接 `to_excel`，**顏色和格式會一起帶過去**（需要 openpyxl 套件）。

In [6]:
with pd.ExcelWriter('空氣品質週報.xlsx') as writer:
    styled.to_excel(writer, sheet_name='本週AQI')
    report.to_excel(writer, sheet_name='年度成績單')
    heat_style.to_excel(writer, sheet_name='月份熱力圖')
print('已匯出 空氣品質週報.xlsx（用 Excel 打開，顏色都還在）')

已匯出 空氣品質週報.xlsx（用 Excel 打開，顏色都還在）


---

## ✍️ 換你動手：練習區

每一題都有三格：**① 半成品程式**（把 `___` 換成正確的程式）→ **② ✅ 檢查格**（自動告訴你對不對）→ **③ 🔑 參考解答**（折疊起來，先自己試）。

> 看到 `NameError: name '___' is not defined` 不用緊張，那代表還有空格沒填。

In [7]:
# @title 🧰 練習檢查小幫手（執行一次就好，不用看懂裡面的程式）
import math
import numpy as np
import pandas as pd


def _same(ans, expected, ordered=False):
    if isinstance(expected, (pd.DataFrame, pd.Series)):
        if type(ans) is not type(expected):
            return False
        try:
            assert_equal = pd.testing.assert_frame_equal if isinstance(expected, pd.DataFrame) else pd.testing.assert_series_equal
            if not ordered:
                ans, expected = ans.sort_index(), expected.sort_index()
            assert_equal(ans, expected, check_names=False, check_dtype=False,
                         check_index_type=False, check_exact=False, rtol=1e-3, atol=0.06,
                         **({'check_column_type': False} if isinstance(expected, pd.DataFrame) else {}))
            return True
        except (AssertionError, TypeError, ValueError):
            return False
    if isinstance(expected, np.ndarray):
        return np.asarray(ans).shape == expected.shape and np.allclose(np.asarray(ans, dtype=float), expected, atol=0.06)
    if isinstance(expected, (int, float, np.number)) and not isinstance(expected, bool):
        try:
            return math.isclose(float(ans), float(expected), rel_tol=1e-3, abs_tol=0.06)
        except (TypeError, ValueError):
            return False
    if isinstance(expected, (list, tuple, set)):
        return list(ans) == list(expected) if not isinstance(expected, set) else set(ans) == expected
    return ans == expected


def check(name, expected, ordered=False):
    """比對學生的變數 name 和正確答案 expected，印出 ✅ 或 ❌；ordered=True 時連排列順序也要一樣。"""
    if name not in globals():
        print(f'⏳ 還找不到變數 {name}：請先把上一格的 ___ 填好並執行')
        return
    ans = globals()[name]
    if _same(ans, expected, ordered):
        print(f'✅ {name} 答對了！')
        return
    print(f'❌ {name} 還不對，再檢查看看：')
    if type(ans) is not type(expected):
        print(f'   型態不同：你的是 {type(ans).__name__}，應該是 {type(expected).__name__}')
    elif hasattr(expected, 'shape') and getattr(ans, 'shape', None) != expected.shape:
        print(f'   大小不同：你的是 {ans.shape}，應該是 {expected.shape}')
    else:
        print('   型態和大小都對了，是數值' + ('或排列順序' if ordered else '') + '不一樣——檢查一下篩選條件或計算方式')


print('🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格')

🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格


### ✍️ 練習 1：在成績單上，把「平均 AQI」超過全部測站平均的格子改成粗體紅字

<details><summary>💡 卡住了？點我看提示</summary>

先算出 `year['平均AQI'].mean()`。`report.map(函式, subset=['平均AQI'])` 的函式要回傳 CSS，例如 `'color: red; font-weight: bold'`，不符合時回傳 `None`。

</details>

In [ ]:
# ✏️ 練習 1：把 ___ 換成正確的程式，再按執行
avg = year['平均AQI'].___()
ex1 = report.map(lambda v: 'color: red; font-weight: bold' if v > ___ else None, subset=['平均AQI'])
ex1

In [ ]:
# @title ✅ 執行這格，檢查練習 1
check('avg', year['平均AQI'].mean())

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
avg = year['平均AQI'].mean()
ex1 = report.map(lambda v: 'color: red; font-weight: bold' if v > avg else None, subset=['平均AQI'])
ex1
```

</details>

### ✍️ 練習 2：公布欄版本：隱藏「最差AQI」欄位

<details><summary>💡 卡住了？點我看提示</summary>

`.hide(subset=[欄位], axis='columns')` 只是「不顯示」，資料本身還在。

</details>

In [ ]:
# ✏️ 練習 2：把 ___ 換成正確的程式，再按執行
ex2 = report.hide(subset=[___], axis='columns')
ex2

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
ex2 = report.hide(subset=['最差AQI'], axis='columns')
ex2
```

</details>

### ✍️ 練習 3：熱力圖改成 `axis=1`：顏色的意思變成什麼？

<details><summary>💡 卡住了？點我看提示</summary>

`axis=None` 是整張表一起比；`axis=1` 是「每一列（每個測站）自己比」——看的是「這個測站哪個月最糟」，就算是空氣很好的花蓮，也會有自己的紅色月份。

</details>

In [ ]:
# ✏️ 練習 3：把 ___ 換成正確的程式，再按執行
ex3 = heat.style.background_gradient(cmap='YlOrRd', axis=___).format('{:.0f}')
ex3

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
ex3 = heat.style.background_gradient(cmap='YlOrRd', axis=1).format('{:.0f}')
ex3
# axis=1 → 每個測站各自上色：看的是「對這個測站來說，哪個月最糟」，而不是「哪個測站最糟」
```

</details>

---

## 🏆 任務完成！今天你用程式做到了這些事

- ✅ 任務 1：先做出「素顏」的週報
- ✅ 任務 2：套上官方的 AQI 顏色
- ✅ 任務 3：年度成績單——格式化、資料條、標出最好與最差
- ✅ 任務 4：熱力圖——一年 12 個月，哪裡、什麼時候最糟？
- ✅ 任務 5：匯出成 Excel，貼到公布欄

> 🎯 **帶走一句話**：程式只是工具，真正有價值的是你從資料裡找到的那句「所以我們應該……」。
> 把上面每個 💡 **看懂結果** 用自己的話寫成一句建議，就是一份可以拿去報告的分析。